# Class 1: Text Search & Clustering
## Learning Notebook Part 1 - Foundation: Preprocessing & Bag of Words

**Welcome!** This notebook focuses on **preprocessing and tokenization** - the foundation of all NLP work!

**This notebook will:**
- 📝 Teach text preprocessing with **interactive TODOs** for class participation
- 🔧 Show regex patterns and tokenization techniques step-by-step  
- 💡 Explain concepts with examples (complete implementations are in Exercise Notebook)

**For hands-on practice:**
- Complete implementations and exercises are in the **Exercise Notebook**

Let's start by understanding why text preprocessing matters!
- Convert text into numbers that computers can understand

**By the end**: You'll understand text preprocessing, Bag of Words (which equals Term Frequency), and how to convert text to numbers. You'll learn that BOW is **syntactic** (word-based, no meaning) - true semantic search (understanding meaning) comes in Class 2 with embeddings!

**Important**: **Semantic = meaning**. In this class, we learn **syntactic** models (BOW/TF) that work with word counts/frequencies but don't understand meaning. Semantic models (embeddings) that understand meaning are in Class 2!

---

## Today's Goal: Building a Movie Search System

**The Problem**: You're building a movie recommendation system. Users want to:
- 🔍 **Search** for movies by description (e.g., "space adventure", "mind-bending thriller")
- 📊 **Discover** similar movies automatically
- 💡 **Understand** meaning, not just match exact words

**The Challenge**: 
- "Space adventure" should find "cosmic journey" and "galactic exploration" (requires understanding meaning - synonyms!)
- "Mind-bending" should find "psychological thriller" and "complex narrative" (requires understanding meaning!)
- We want to understand **meaning**, not just keywords!

**What we'll learn TODAY (Syntactic approaches)**:
1. Start with simple keyword search (see its limitations)
2. Learn text preprocessing (cleaning, tokenization, regex)
3. Learn n-grams (feature extraction - capturing word order)
4. Convert text to numbers (Bag of Words = Term Frequency - vectorization)
5. **Next in Part 2**: TF-IDF, similarity-based search, and clustering

**Pipeline order** (very important!):
```
Preprocessing → Tokenization → N-grams → Vectorization (BoW/TF) → Applications (Search, Clustering)
```

**What's coming NEXT CLASS (Semantic approaches)**:
- Embeddings for true semantic search (understanding meaning, synonyms)
- "Space" and "cosmic" will be similar because they share meaning!

**Key Point**: In this class, we learn **syntactic** models (BOW, TF-IDF) - they work with word counts but don't understand meaning. **Semantic = meaning**. True semantic search comes in Class 2!

**This is top-down learning**: We'll see the problem first, then learn the tools to solve it!

---

## What is Natural Language Processing (NLP)?

**NLP** = Teaching computers to understand, interpret, and generate human language

### Real-World Applications (Why This Matters!)

| Application | Example | Why It's Important |
|------------|---------|-------------------|
| **Search Engines** | Google, Bing | Finding relevant results for your queries |
| **Virtual Assistants** | Siri, Alexa, ChatGPT | Understanding what you're asking |
| **Spam Detection** | Email filters | Automatically identifying unwanted emails |
| **Sentiment Analysis** | Review analysis | Understanding if reviews are positive/negative |
| **Translation** | Google Translate | Converting between languages |
| **Text Summarization** | News digests | Condensing long articles into key points |

**Today's focus**: Search and clustering - foundational NLP tasks you'll use everywhere!

---

## Machine Learning in NLP: Two Approaches

### Unsupervised Learning (Today's Focus!)
- ❌ **No labels needed** - we don't tell the model the "right" answer
- ✅ **Clustering**: Automatically finding groups (e.g., similar movies)
- ✅ **Search**: Finding similar documents without examples
- 🎯 **Goal**: Discover patterns in the data

### Supervised Learning (You Know This from Chapter 0!)
- ✅ **Needs labeled data** - we provide correct answers
- 📊 **Text Classification**: Spam/not spam, genre classification
- 💭 **Sentiment Analysis**: Positive/negative/neutral labels
- 🎯 **Goal**: Learn to predict labels from examples

**Key Insight**: Same preprocessing, same vectors - just different goals!
- Unsupervised: Find patterns (no labels)
- Supervised: Predict labels (with examples)

We'll connect these at the end!


## Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# For better output display
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"


### Load the Data


In [ ]:
# Load movie descriptions
df = pd.read_csv('data/movies.csv')
print(f"Loaded {len(df)} movies")
df.head()


## Why Text is Hard

Computers work with numbers, but text is made of words. This creates several challenges:

1. **Unstructured**: Text doesn't have a fixed format
2. **Synonyms**: "space" vs "cosmic" vs "galactic" - different words, similar meaning
3. **Context**: "bank" could mean financial institution or river edge
4. **Variations**: "sci-fi", "science fiction", "Science Fiction" - same concept
5. **Word order matters**: "dog bites man" vs "man bites dog" - completely different!

Let's look at our movie descriptions:


In [ ]:
# Let's examine a movie description
print("Movie Description Example:")
print("=" * 60)
print(f"Title: {df.loc[0, 'title']}")
print(f"Description: {df.loc[0, 'description']}")
print("=" * 60)


## Two Approaches to Search

### 1. Keyword Search (Simple but Limited)
- Looks for exact word matches
- Can use simple substring matching or Term Frequency (TF) approaches
- Fast and simple
- Fails with synonyms ("space movie" won't find "cosmic adventure")
- **Syntactic only** - works with words, not meaning

### 2. True Semantic Search (Class 2 - Embeddings!)
- **Semantic = meaning-based** - understands synonyms and related concepts
- "Space" and "cosmic" are close in meaning (semantic similarity)
- Requires embeddings (dense vectors that capture meaning)
- This is what we'll learn in Class 2!

**Key distinction:**
- **Syntactic models** (BOW/TF): Work with word presence/frequency, no understanding of meaning
- **Semantic models** (embeddings): Understand meaning - synonyms and related concepts are similar

Let's start with different keyword search approaches:


### Approach 1: Simple Substring Matching (Vanilla Keyword Search)

The most basic approach - just check if the query word appears in the text:


In [ ]:
def simple_keyword_search(df, query, column='description'):
    """
    Simple keyword search: finds documents containing the query words (exact match)
    """
    query_lower = query.lower()
    results = []
    
    for idx, row in df.iterrows():
        text = str(row[column]).lower()
        if query_lower in text:
            results.append({
                'movie_id': row['movie_id'],
                'title': row['title'],
                'match': True
            })
    
    return pd.DataFrame(results)

# Search for "space"
results = simple_keyword_search(df, "space")
print(f"Found {len(results)} results for 'space':")
results


### Approach 2: Term Frequency (TF) Based Keyword Search

Instead of just checking if a word exists, we count how many times it appears and rank results by frequency:


In [ ]:
def calculate_term_frequency(text, word):
    """
    Calculate Term Frequency: how many times a word appears in a text
    """
    text_lower = str(text).lower()
    word_lower = word.lower()
    
    # Simple word count (tokenize by splitting on whitespace)
    words = text_lower.split()
    count = words.count(word_lower)
    
    # TF = count / total_words (normalized)
    total_words = len(words)
    tf = count / total_words if total_words > 0 else 0
    
    return count, tf

# Example
text = df.loc[5, 'description']  # Interstellar - about space
print(f"Text: {text[:100]}...")
print(f"Count of 'space': {calculate_term_frequency(text, 'space')[0]}")
print(f"TF of 'space': {calculate_term_frequency(text, 'space')[1]:.4f}")


In [ ]:
def tf_keyword_search(df, query, column='description', top_k=5):
    """
    Keyword search using Term Frequency - ranks results by how often query words appear
    """
    query_lower = query.lower()
    query_words = query_lower.split()
    
    results = []
    
    for idx, row in df.iterrows():
        text = str(row[column]).lower()
        
        # Calculate total TF score across all query words
        total_tf = 0
        word_count = 0
        
        for word in query_words:
            count, tf = calculate_term_frequency(text, word)
            if count > 0:  # Only count if word appears
                total_tf += tf
                word_count += count
        
        if word_count > 0:  # At least one query word found
            results.append({
                'movie_id': row['movie_id'],
                'title': row['title'],
                'tf_score': total_tf,
                'word_count': word_count
            })
    
    # Sort by TF score (highest first) and return top_k
    results_df = pd.DataFrame(results)
    if len(results_df) > 0:
        results_df = results_df.sort_values('tf_score', ascending=False).head(top_k)
    
    return results_df

# Search for "space adventure"
results = tf_keyword_search(df, "space adventure")
print(f"Found {len(results)} results for 'space adventure' (TF-ranked):")
results


**Key Insight**: TF-based search is better than simple substring matching because:
- It **ranks** results by relevance (more frequent = more relevant)
- It can handle **multiple words** in the query
- It's still fast and simple

**But it still has limitations**:
- "space" won't match "cosmic" or "galactic" (synonyms) - **syntactic only, no meaning**
- "adventure" won't match "journey" or "quest" - different words = zero similarity
- It doesn't understand meaning - it's a **syntactic model** (word-based, not meaning-based)

To do better, we need to:
1. **Preprocess** the text properly (tokenization, normalization)
2. **Convert** text to numerical vectors (Bag of Words = Term Frequency)
3. **Measure similarity** between vectors (we'll learn this in Part 2!)

This moves us to **vectorization** - converting text to numbers. In Part 2, we'll learn TF-IDF for better search and clustering!


## Text Preprocessing Pipeline

Before we can work with text, we need to clean and prepare it. The pipeline has three main stages:

1. **Pre-processing** (before tokenization): Clean the raw text
2. **Tokenization**: Split text into individual words/tokens
3. **Post-processing** (after tokenization): Further refine the tokens

### Why Preprocessing Matters

**Garbage In = Garbage Out**: If we don't clean our text properly, our models will learn from noise, not signal!


### Stage 1: Pre-processing (Before Tokenization)

**Goal**: Clean the raw text before splitting it into words

Common tasks:
- Remove HTML tags
- Remove special characters
- Normalize URLs, emails, phone numbers (using **Regular Expressions/Regex**)
- Handle case (convert to lowercase)
- Remove extra whitespace

Let's see an example with **Regular Expressions (Regex)**:


In [ ]:
# Example text with various issues
sample_text = "Contact us at info@example.com or call (555) 123-4567. Visit https://example.com for more info!!!"
print("Original:", sample_text)

# TODO (Together): Let's remove URLs using regex
# Pattern: r'https?://\S+' matches http:// or https:// followed by non-whitespace
# Try to understand the pattern, then we'll run it together!
text_no_urls = re.sub(r'https?://\S+', '', sample_text)
print("\nAfter removing URLs:", text_no_urls)

# TODO (Together): Remove email addresses
# Can you think of a pattern for emails? Hint: something@something
# Let's discuss and then implement:
text_no_emails = re.sub(r'\S+@\S+', '', text_no_urls)
print("After removing emails:", text_no_emails)

# TODO (Together): Remove phone numbers
# Phone numbers can be: (555) 123-4567 or 555-123-4567
# Try creating a pattern that handles both formats!
text_no_phones = re.sub(r'\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}', '', text_no_emails)
print("After removing phones:", text_no_phones)

# TODO (Together): Remove punctuation but keep letters, numbers, spaces
# Pattern hint: [^\w\s] means "not word characters or whitespace"
text_clean = re.sub(r'[^\w\s]', ' ', text_no_phones)
print("After removing punctuation:", text_clean)

# TODO (Together): Normalize whitespace and lowercase
# Pattern: r'\s+' matches one or more whitespace characters
text_final = re.sub(r'\s+', ' ', text_clean).strip().lower()
print("Final (lowercase, normalized):", text_final)


### Stage 2: Tokenization

**Goal**: Split text into individual words (tokens)

Tokens can be:
- Words: "machine", "learning"
- Punctuation: ".", ","
- Numbers: "2024"
- Subwords: "un-" + "happiness" (advanced)

Simple approach: split on whitespace
Better approach: use regex to find word boundaries


In [ ]:
# Simple tokenization examples
text = "Natural Language Processing is amazing! It's used everywhere."
print("Original:", text)

# Split on whitespace (simple but loses punctuation)
tokens_simple = text.split()
print("\nSimple split:", tokens_simple)

# Regex tokenization (find all word characters)
tokens_regex = re.findall(r'\w+', text.lower())
print("Regex tokenization:", tokens_regex)


### Stage 3: Post-processing (After Tokenization)

**Goal**: Further refine tokens by removing noise

Common tasks:
- Remove **stop words**: "the", "a", "an", "and", "or" (common but not informative)
- Remove very short tokens: "I", "a" (often noise)
- Stemming/Lemmatization: "running" → "run" (we'll skip this for now)


In [ ]:
# Common stop words
STOP_WORDS = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 
              'of', 'with', 'by', 'is', 'are', 'was', 'were', 'be', 'been', 'being',
              'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could',
              'should', 'may', 'might', 'must', 'can', 'this', 'that', 'these', 'those',
              'i', 'you', 'he', 'she', 'it', 'we', 'they', 'what', 'which', 'who'}

def preprocess_text(text):
    """
    Complete preprocessing pipeline: clean, tokenize, filter
    Let's build this together step by step!
    """
    # Step 1: Pre-processing - lowercase and normalize
    text = str(text).lower()
    text = re.sub(r'\s+', ' ', text).strip()
    
    # TODO (Together): Step 2 - Tokenization
    # Use regex (optionally) to find all words. What pattern would extract words?
    # Hint: \w+ matches word characters
    tokens = re.findall(r'\w+', text)
    
    # TODO (Together): Step 3 - Post-processing
    # Remove stop words and very short tokens (length < 3)
    # Can you write the list comprehension?
    tokens_clean = [token for token in tokens 
                    if token not in STOP_WORDS and len(token) > 2]
    
    return tokens_clean

# Example
sample = df.loc[0, 'description']
print("Original:", sample)
print("\nPreprocessed tokens:", preprocess_text(sample))


---

## N-grams: Capturing Word Order (Feature Extraction)

**After preprocessing and tokenization**, we can create **n-grams** - sequences of n consecutive tokens. N-grams help capture some word order information that simple Bag of Words loses!

### What are N-grams?

**N-grams** = Sequences of N consecutive tokens (words or characters)

**Types:**
- **Unigrams (1-grams)**: Single words - "I", "love", "Python"
- **Bigrams (2-grams)**: Pairs of consecutive words - "I love", "love Python"
- **Trigrams (3-grams)**: Triples of consecutive words - "I love Python"

### Why N-grams Matter

**Problem with unigrams (single words)**: "Dog bites man" and "Man bites dog" have the same unigrams!

**Solution with bigrams**: 
- "Dog bites man" → ["dog bites", "bites man"]
- "Man bites dog" → ["man bites", "bites dog"]
- Different bigrams → different meaning captured!

### Example:

```
Text: "I love Python programming"

Unigrams (1-grams):
  ["i", "love", "python", "programming"]

Bigrams (2-grams):
  ["i love", "love python", "python programming"]

Trigrams (3-grams):
  ["i love python", "love python programming"]
```

**Trade-off**: 
- ✅ Better capture of word order and context
- ❌ Larger vocabulary (more features)
- ❌ Can be sparse (many n-grams appear only once)

**Common Usage**: Combining unigrams + bigrams gives a good balance!


In [ ]:
# Example: Creating n-grams from tokens

def create_ngrams(tokens, n=2):
    """
    Create n-grams from a list of tokens.
    
    Args:
        tokens: List of tokens (words)
        n: Size of n-gram (1=unigrams, 2=bigrams, 3=trigrams)
    
    Returns:
        list: List of n-grams
    """
    if n == 1:
        return tokens
    else:
        ngrams = []
        for i in range(len(tokens) - n + 1):
            # Create n-gram by joining n consecutive tokens
            ngram = ' '.join(tokens[i:i+n])
            ngrams.append(ngram)
        return ngrams

# Example text
text = "I love Python programming"
tokens = preprocess_text(text)
print(f"Original text: {text}")
print(f"Tokens: {tokens}\n")

# Create different n-grams
unigrams = create_ngrams(tokens, n=1)
bigrams = create_ngrams(tokens, n=2)
trigrams = create_ngrams(tokens, n=3)

print("Unigrams (1-grams):")
print(f"  {unigrams}\n")

print("Bigrams (2-grams):")
print(f"  {bigrams}\n")

print("Trigrams (3-grams):")
print(f"  {trigrams}\n")

# Compare: Same words, different order
text1 = "Dog bites man"
text2 = "Man bites dog"

tokens1 = preprocess_text(text1)
tokens2 = preprocess_text(text2)

bigrams1 = create_ngrams(tokens1, n=2)
bigrams2 = create_ngrams(tokens2, n=2)

print("=" * 60)
print("Comparing word order with bigrams:")
print("=" * 60)
print(f"Text 1: '{text1}'")
print(f"  Bigrams: {bigrams1}")
print(f"\nText 2: '{text2}'")
print(f"  Bigrams: {bigrams2}")
print(f"\nDifferent bigrams = different meaning captured!")


### N-grams in Production

In practice, you'll use libraries like scikit-learn's `CountVectorizer` or `TfidfVectorizer` which can automatically create n-grams:

```python
# Example (we'll see this in Part 2):
from sklearn.feature_extraction.text import TfidfVectorizer

# Create unigrams and bigrams
vectorizer = TfidfVectorizer(ngram_range=(1, 2))  # Unigrams + bigrams
```

**Common choices:**
- `ngram_range=(1, 1)`: Only unigrams (standard BoW/TF)
- `ngram_range=(1, 2)`: Unigrams + bigrams (good balance)
- `ngram_range=(2, 2)`: Only bigrams
- `ngram_range=(1, 3)`: Unigrams + bigrams + trigrams (more features, can be sparse)

**Key Insight**: N-grams are created **after tokenization** and **before vectorization** (BoW/TF). They help capture word order, but still don't capture semantic meaning - that requires embeddings (Class 2)!


---

## From Text to Numbers: Bag of Words (BoW) - Vectorization

**Now that we have preprocessed tokens (and optionally n-grams)**, we need to convert them to numbers that computers can work with. This is called **vectorization**.

**Pipeline reminder:**
1. ✅ Preprocessing (clean, normalize)
2. ✅ Tokenization (split into words)
3. ✅ Post-processing (filter, remove stop words)
4. ✅ Feature extraction (n-grams - optional, we just covered this!)
5. **← We are here**: Vectorization (Bag of Words = Term Frequency)
6. Applications (Similarity Search, Clustering - Part 2)

### The Bag of Words Model (BoW = Term Frequency)

**Important**: **Bag of Words (BoW) = Term Frequency (TF)** - they are the same thing!

**Idea**: Represent each document as a vector of word counts, ignoring word order.

The process:
1. Create vocabulary (list of all unique words from the corpus)
2. Count how many times each word appears in a document (this is Term Frequency!)
3. Represent document as vector of counts

**Terminology reminder:**
- **Corpus**: Collection of all documents we're working with
- **Vocabulary**: All unique words/tokens in the corpus
- **Token**: Individual word after tokenization

Example:
- Document 1: "I love Python"
- Document 2: "Python is great"
- **Corpus**: The collection of both documents
- **Vocabulary**: ["i", "love", "python", "is", "great"] (all unique words from the corpus)

**Bag of Words vectors**:
- Doc 1: [1, 1, 1, 0, 0]  (one "i", one "love", one "python")
- Doc 2: [0, 0, 1, 1, 1]  (one "python", one "is", one "great")

**Key insight**: We've lost word order! "Dog bites man" = "Man bites dog" in BoW. But for many tasks, this is okay!

**Remember**: BoW = TF (Term Frequency) - it's just counting word frequencies!


In [ ]:
# Simple Bag of Words example
# Our corpus: a collection of documents
docs = [
    "I love Python programming",
    "Python is a programming language",
    "I love machine learning"
]

# Step 1: Build vocabulary from the corpus (all unique words)
all_words = set()
for doc in docs:
    tokens = preprocess_text(doc)
    all_words.update(tokens)

vocab = sorted(list(all_words))
print(f"Corpus: {len(docs)} documents")
print("Vocabulary:", vocab)
print(f"Vocabulary size: {len(vocab)}")

# Create BoW vectors
bow_vectors = []
for doc in docs:
    tokens = preprocess_text(doc)
    word_counts = Counter(tokens)
    vector = [word_counts.get(word, 0) for word in vocab]
    bow_vectors.append(vector)
    print(f"\n'{doc}' -> {vector}")

print("\nBag of Words Matrix:")
print(pd.DataFrame(bow_vectors, columns=vocab))


## Sparse vs Dense Vectors

This is a crucial concept in NLP!

### Sparse Vectors (like BoW)
- **Most values are zero** (e.g., [0, 0, 1, 0, 0, 0, 0, 1, 0, 0, ...])
- Vocabulary size can be huge (10,000-100,000+ words)
- Each document only uses a small fraction of the vocabulary
- **Memory efficient** when stored in sparse format (only store non-zero values)
- Example: Bag of Words, TF-IDF

### Dense Vectors (like Embeddings - we'll see this next class!)
- **Most/all values are non-zero** (e.g., [0.23, -0.15, 0.87, ..., 0.42])
- Fixed, smaller dimension (typically 100-768 dimensions)
- Each dimension has meaning (learned representation)
- **Captures relationships** between words
- Example: Word embeddings, sentence embeddings

Let's visualize this:


## Comparing: With vs Without Preprocessing

**Key Question**: Does preprocessing make a difference? Let's find out!


In [ ]:
# Comparison: BoW with preprocessing vs without preprocessing
# Our corpus: collection of sample documents
docs_sample = [
    "I LOVE Python! It's amazing!!!",
    "Python is a programming language.",
    "i love machine learning!"
]

print("=" * 70)
print("WITHOUT Preprocessing (raw text):")
print("=" * 70)
print(f"Corpus: {len(docs_sample)} documents\n")

# BoW without preprocessing - just split on whitespace
# Build vocabulary from corpus
all_words_no_preprocess = set()
for doc in docs_sample:
    words = doc.lower().split()  # Simple split, no cleaning
    all_words_no_preprocess.update(words)

vocab_no_preprocess = sorted(list(all_words_no_preprocess))
print(f"Vocabulary: {vocab_no_preprocess}")
print(f"Vocabulary size: {len(vocab_no_preprocess)}")
print(f"\nNotice: 'i', 'python!', 'it's', 'amazing!!!', 'language.' are separate words!")
print(f"Punctuation and case variations create different words!")

bow_no_preprocess = []
for doc in docs_sample:
    words = doc.lower().split()
    word_counts = Counter(words)
    vector = [word_counts.get(word, 0) for word in vocab_no_preprocess]
    bow_no_preprocess.append(vector)

print("\nBoW Matrix (no preprocessing):")
print(pd.DataFrame(bow_no_preprocess, columns=vocab_no_preprocess, 
                   index=[f"Doc {i+1}" for i in range(len(docs_sample))]))

print("\n" + "=" * 70)
print("WITH Preprocessing (cleaned and tokenized):")
print("=" * 70)
print(f"Corpus: {len(docs_sample)} documents\n")

# BoW with preprocessing
# Build vocabulary from corpus (after preprocessing)
all_words_preprocess = set()
for doc in docs_sample:
    tokens = preprocess_text(doc)
    all_words_preprocess.update(tokens)

vocab_preprocess = sorted(list(all_words_preprocess))
print(f"Vocabulary: {vocab_preprocess}")
print(f"Vocabulary size: {len(vocab_preprocess)}")
print(f"\nNotice: Clean words only! Punctuation removed, stop words filtered!")

bow_preprocess = []
for doc in docs_sample:
    tokens = preprocess_text(doc)
    word_counts = Counter(tokens)
    vector = [word_counts.get(word, 0) for word in vocab_preprocess]
    bow_preprocess.append(vector)

print("\nBoW Matrix (with preprocessing):")
print(pd.DataFrame(bow_preprocess, columns=vocab_preprocess,
                   index=[f"Doc {i+1}" for i in range(len(docs_sample))]))

print("\n" + "=" * 70)
print("KEY INSIGHT:")
print("=" * 70)
print("Without preprocessing: More vocabulary words, many variations of same word")
print("With preprocessing: Cleaner vocabulary, focuses on meaningful words")
print("→ Preprocessing reduces noise and makes patterns clearer!")


## The Problem with Syntactic Representations (BoW Limitations)

**Critical Understanding**: Bag of Words is a **syntactic representation** (and so is TF-IDF) - they only capture word presence/frequency, NOT meaning!

**Key Terminology:**
- **Syntactic**: Based on word structure/frequency (BOW, TF-IDF) - no understanding of meaning
- **Semantic**: Based on meaning - understands synonyms and related concepts (embeddings - Class 2)

**Remember**: Semantic = meaning. Syntactic models like BOW and TF-IDF do NOT have meaning - they work with word counts/frequencies only!

Let's see the issues:


In [ ]:
# Demonstrating BoW/Syntactic Representation Limitations

print("=" * 70)
print("Problem 1: Word Order is Lost")
print("=" * 70)

# Corpus: two documents with different word order
docs_order = [
    "The dog bites the man",
    "The man bites the dog"
]

# TODO (Together): Create BoW vectors for these two sentences
# What do you notice about the vectors?
# Build vocabulary from corpus
all_words_order = set()
for doc in docs_order:
    tokens = preprocess_text(doc)
    all_words_order.update(tokens)

vocab_order = sorted(list(all_words_order))
print(f"Vocabulary: {vocab_order}")

bow_order = []
for doc in docs_order:
    tokens = preprocess_text(doc)
    word_counts = Counter(tokens)
    vector = [word_counts.get(word, 0) for word in vocab_order]
    bow_order.append(vector)
    print(f"\n'{doc}'")
    print(f"→ {vector}")

print("\n" + "=" * 50)
print("Result: Both sentences have IDENTICAL BoW vectors!")
print("But they have COMPLETELY DIFFERENT meanings!")
print("=" * 50)

print("\n" + "=" * 70)
print("Problem 2: Synonyms are Treated as Completely Different")
print("=" * 70)

# Corpus: documents with synonyms
docs_synonyms = [
    "I love machine learning",
    "I adore artificial intelligence"
]

# TODO (Together): Create BoW vectors for these
# Build vocabulary from corpus
all_words_syn = set()
for doc in docs_synonyms:
    tokens = preprocess_text(doc)
    all_words_syn.update(tokens)

vocab_syn = sorted(list(all_words_syn))
print(f"Vocabulary: {vocab_syn}")

bow_syn = []
for doc in docs_synonyms:
    tokens = preprocess_text(doc)
    word_counts = Counter(tokens)
    vector = [word_counts.get(word, 0) for word in vocab_syn]
    bow_syn.append(vector)
    print(f"\n'{doc}'")
    print(f"→ {vector}")

print("\n" + "=" * 50)
print("Result: 'love' vs 'adore' and 'machine learning' vs 'AI' are completely different!")
print("But they have SIMILAR meanings!")
print("→ BoW similarity = 0 even though meanings are related")
print("=" * 50)

print("\n" + "=" * 70)
print("Problem 3: Context is Lost")
print("=" * 70)

# Corpus: documents with same word but different context
docs_context = [
    "The bank is closed",      # Financial bank
    "The river bank is muddy"  # River edge
]

# Build vocabulary from corpus
all_words_ctx = set()
for doc in docs_context:
    tokens = preprocess_text(doc)
    all_words_ctx.update(tokens)

vocab_ctx = sorted(list(all_words_ctx))
print(f"Vocabulary: {vocab_ctx}")

bow_ctx = []
for doc in docs_context:
    tokens = preprocess_text(doc)
    word_counts = Counter(tokens)
    vector = [word_counts.get(word, 0) for word in vocab_ctx]
    bow_ctx.append(vector)
    print(f"\n'{doc}'")
    print(f"→ {vector}")

print("\n" + "=" * 50)
print("Result: 'bank' appears in both, but means completely different things!")
print("→ BoW can't distinguish context/meaning")
print("=" * 50)

print("\n" + "=" * 70)
print("SUMMARY: Syntactic Representation (BoW/TF-IDF) Limitations")
print("=" * 70)
print("❌ Loses word order")
print("❌ Can't handle synonyms (different words = 0 similarity)")
print("❌ Can't understand context")
print("❌ Only captures word frequency, NOT meaning")
print("\n📌 CRITICAL: Syntactic ≠ Semantic")
print("   - Syntactic (BOW/TF-IDF): Word-based, no meaning")
print("   - Semantic (Embeddings - Class 2): Meaning-based, understands synonyms")
print("\n✅ But syntactic models are simple, interpretable, and work for many tasks!")
print("\n💡 Next class: We'll see embeddings (semantic representations) that solve these!")
print("   - Semantic = meaning - embeddings understand that 'space' and 'cosmic' are similar!")


---

## End of Part 1: Foundation Complete! 🎉

**Great job!** You've learned the fundamentals:
- ✅ **Text preprocessing** (cleaning, tokenization, regex)
- ✅ **Converting text to numbers** (Bag of Words = Term Frequency - vectorization)
- ✅ **Understanding sparse vectors**
- ✅ **Recognizing BoW/TF limitations** (syntactic, not semantic)

**The Complete NLP Pipeline You Now Understand:**

```
Raw Text
  ↓
1. Preprocessing (clean, normalize)
  ↓
2. Tokenization (split into words)
  ↓
3. Post-processing (filter stop words)
  ↓
4. Vectorization (Bag of Words = Term Frequency) ← You learned this!
  ↓
5. TF-IDF (weighted vectors) ← Part 2
  ↓
6. Similarity Search & Clustering ← Part 2
```

**Key Point**: **Bag of Words (BoW) = Term Frequency (TF)** - they're the same thing! It's just counting how many times each word appears.

**Now it's time to practice!** 🏋️

Complete the exercises in the **Exercise Notebook** to reinforce what you've learned:
- **Exercise 1**: Text cleaning with regex (URLs, emails, phone numbers)
- **Exercise 2**: Tokenization with stop word removal
- **Exercise 3**: TF-IDF calculation from scratch (TF = BoW!)
- **Exercise 4**: TF-based keyword search implementation
- **Exercise 5**: Similarity-based search with TF-IDF (still syntactic, not semantic!)
- **Exercise 6**: Document clustering with K-Means
- **Exercise 7**: Comparing preprocessing approaches
- **Exercise 8**: Stemming and Lemmatization (advanced preprocessing)

**Next in Part 2**: We'll learn TF-IDF (improving on BoW/TF), similarity search, and clustering!

Take a break, complete the exercises, then continue with **Learning Notebook Part 2** 👉